# 05 — Development experiment: repair candidate consistency

Compare **original CAA**, **budget-only block rollback**, and **relationship + budget block rollback** on a separate development subset. This tests whether mutually consistent feature changes improve valid attack coverage and evasion before developing the IDS defense.

**CPU runtime → Runtime → Run all. No previous ZIP upload is needed.** At the end, download and share the new results ZIP. The default runs 32 development records, two existing models and three attack seeds. It runs six original attacks, so it takes substantially longer than notebook 04.

The original constraints, zero relationship tolerance, checkpoints and scaled L2 radius remain fixed. All previous notebooks and files are preserved in their existing locations.


## 1. Settings and separate workspace

Keep the defaults for the planned experiment. Both repair methods reuse each original attack's exact stage outputs. Extra repair computation and model evaluations are measured. This is a development comparison of attack outputs; it does not train a new IDS.


In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

DEVELOPMENT_RECORDS = 32 #@param {type:"integer"}
SELECTION_SEED = 0 #@param {type:"integer"}
ATTACK_SEEDS = "0,1,2" #@param {type:"string"}
ATTACK_STEPS = 10 #@param {type:"integer"}
GENERATIONS = 100 #@param {type:"integer"}
OFFSPRING = 50 #@param {type:"integer"}
EPSILON = 0.5 #@param {type:"number"}
SEEDS = [int(value.strip()) for value in ATTACK_SEEDS.split(",") if value.strip()]
if not SEEDS or len(set(SEEDS)) != len(SEEDS) or min(SEEDS + [SELECTION_SEED]) < 0:
    raise ValueError("Enter distinct nonnegative attack seeds, separated by commas.")
if min(DEVELOPMENT_RECORDS, ATTACK_STEPS, GENERATIONS, OFFSPRING) < 1 or EPSILON <= 0:
    raise ValueError("Counts and epsilon must be positive.")
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
ROOT = Path(os.environ.get("PHD_WORKSPACE", "/content/MyPhDThesis_DevelopmentRepair")).resolve()
HOST_PYTHON, HOST_VERSION = sys.executable, sys.version
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} is not a git checkout. Choose a new workspace.")
for required in ["scripts/bootstrap.py", "baseline/run_development_repair.py", "notebooks/05_Development_Consistency_Repair.ipynb"]:
    if not (ROOT / required).is_file():
        raise RuntimeError(f"Missing {required}. Use a fresh clone of the updated repository.")

# Existing checkouts are preserved; no pull/reset/checkout is performed.
def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

SESSION = ROOT / "runs" / unique_name("development_repair")
SESSION.mkdir(parents=True, exist_ok=False)
source_paths = [p for folder in ["baseline", "scripts", "tests"] for p in (ROOT / folder).glob("*") if p.is_file()]
source_paths += [ROOT / "notebooks/05_Development_Consistency_Repair.ipynb", ROOT / "docs/DEVELOPMENT_REPAIR_GUIDE.md"]
context = {
    "project_repository": REPO_URL,
    "project_commit": subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "project_status": subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines(),
    "session_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook_python": HOST_VERSION, "notebook_executable": HOST_PYTHON,
    "source_sha256": {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths},
    "experiment": "development_candidate_consistency_comparison", "evaluation_device": "cpu",
    "development_records": DEVELOPMENT_RECORDS, "selection_seed": SELECTION_SEED,
    "attack_seeds": SEEDS, "steps": ATTACK_STEPS, "generations": GENERATIONS,
    "offspring": OFFSPRING, "epsilon_scaled_l2": EPSILON,
}
(SESSION / "session_context.json").write_text(json.dumps(context, indent=2) + "\n")
for p in source_paths:
    target = SESSION / "source_snapshot" / p.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

def run_command(command, log_name):
    log_path = SESSION / log_name
    if log_path.exists():
        log_path = SESSION / (unique_name(Path(log_name).stem) + ".log")
    with log_path.open("w") as log:
        log.write("Command: " + json.dumps([str(x) for x in command]) + "\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); see {log_path}. Export partial logs with the last cell.")

print("Project commit:", context["project_commit"])
print("New session:", SESSION)


## 2. Prepare the pinned environment

Use the existing isolated Python 3.8.20 / NumPy 1.23.5 / CPU PyTorch setup. The notebook kernel stays unchanged, and a GPU is not needed. Setup also runs the regression tests, including the final validity/score checks.


In [ ]:
run_command([HOST_PYTHON, ROOT / "scripts/bootstrap.py"], "setup.log")
PYTHON = ROOT / ".baseline-env/bin/python"
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION
setup_copy = SESSION / unique_name("setup_report")
shutil.copy2(ROOT / ".cache/setup_report.json", setup_copy.with_suffix(".json"))
print("Pinned audit interpreter:", PYTHON)


## 3. Download and verify the pinned assets

A fresh checkout downloads approximately 348 MB of feature data plus the small checkpoints. All files are checked against pinned SHA-256 values. These are aggregated CTU feature records, not PCAP traffic.

The compatibility scaler uses all feature rows to match these existing checkpoints. The experiment selects and scores only the official validation partition; no native test rows are attacked or scored. New defense training later requires train-only preprocessing.


In [ ]:
run_command([PYTHON, ROOT / "baseline/download_assets.py", "--assets", ROOT / "assets"], "download.log")
manifest_copy = SESSION / (unique_name("download_manifest") + ".json")
shutil.copy2(ROOT / "assets/download_manifest.json", manifest_copy)


## 4. Run the paired development comparison

For each model and attack seed, run the original CAA once. Save its outputs, then apply the two postprocessors to that same pool. The selection is fixed across models and seeds and does not filter by model predictions.

Related mutable features form dependency blocks. Relationship repair restores a failing block to its own original values. Budget rollback restores high-cost blocks until the original radius is satisfied. Every repaired candidate is checked again; changed feasible vectors are rescored. Completely clean fallbacks never count as changed coverage or new evasions.

The original valid evasions remain available to both augmented methods. Progress prints after each model, seed and method.


In [ ]:
RUN = SESSION / unique_name("comparison")
run_command([PYTHON, ROOT / "baseline/run_development_repair.py",
             "--repo", ROOT / "vendor/tabularbench", "--assets", ROOT / "assets",
             "--out", RUN, "--n-dev", DEVELOPMENT_RECORDS,
             "--selection-seed", SELECTION_SEED, "--seeds", *SEEDS,
             "--steps", ATTACK_STEPS, "--generations", GENERATIONS,
             "--offspring", OFFSPRING, "--eps", EPSILON], "comparison.log")
print("Completed development comparison:", RUN)


## 5. Compare coverage, evasion and added cost

**Coverage** counts distinct original records with at least one changed, valid, in-budget candidate. **New evasion rate** uses records initially detected by the model as its denominator. Candidate totals include duplicates, while coverage and evasion are record-level measures.

The mean and sample standard deviation across seeds describe search variability on the same inputs. They are not population confidence intervals. Additional runtime includes postprocessing and evidence writing; forward-row counts include model evaluations, not network traffic requests.


In [ ]:
import pandas as pd
from IPython.display import display

protocol = json.loads((RUN / "protocol.json").read_text())
results = json.loads((RUN / "results.json").read_text())
split = json.loads((RUN / "split_manifest.json").read_text())
assert protocol["status"] == results["status"] == "completed"
assert split["selected_rows_all_in_official_validation"]
assert split["selected_rows_disjoint_from_native_test"]
assert not protocol["test_partition_evaluated"]
comparison = pd.read_csv(RUN / "comparison.csv")
summary = pd.read_csv(RUN / "seed_summary.csv")
print("Selected development records:", split["selected"])
print("Attack seeds:", SEEDS)
print("Native test partition evaluated:", protocol["test_partition_evaluated"])
display(comparison[["model", "seed", "method", "clean_detected", "union_covered_records",
                    "additional_covered_records", "newly_evaded",
                    "new_evasion_rate_among_initially_detected", "pool_invalid_candidates",
                    "added_wall_seconds", "added_prediction_forward_rows"]])
display(summary[["model", "method", "runs", "union_coverage_fraction_mean",
                 "union_coverage_fraction_sample_std", "new_evasion_rate_among_initially_detected_mean",
                 "new_evasion_rate_among_initially_detected_sample_std", "added_wall_seconds_mean"]])
print("Coverage gains and evasion gains are separate outcomes. Inspect both before claiming an improvement.")
print("Full protocol and record-level evidence:", RUN)


## 6. Download the evidence

The new ZIP includes settings, source snapshots, pinned-asset hashes, original and repaired candidate arrays, final selected examples, validity masks, predictions, record-level summaries, timing and forward counts. The large dataset and environment are excluded.

**Share this new ZIP.** If a previous cell fails, run this export cell separately to preserve partial logs. Optional Drive copying is disabled unless you enable it in the settings.


In [ ]:
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
export_base = EXPORTS / unique_name(SESSION.name + "_results")
archive = Path(shutil.make_archive(str(export_base), "zip", root_dir=SESSION.parent, base_dir=SESSION.name))
print("Results ZIP:", archive)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/MyPhDThesis/results")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / archive.name
    if target.exists():
        raise FileExistsError(f"Preserving existing archive: {target}")
    shutil.copy2(archive, target)
    print("Copied results to Google Drive:", target)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))


## Interpret the result before extending the research

- **More valid coverage and more evasion:** investigate the additional exposed weaknesses and the cost of obtaining them.
- **More valid coverage without more evasion:** feasibility improved, but stronger evasion is still unproven; consider how to search within consistent feature groups.
- **Mostly clean fallbacks:** the repair is too conservative for those outputs. Do not count fallbacks as successful adversarial examples.

This implements a simple repair baseline. It is not an optimal projection, a new defense model, or proof of live-traffic realizability. Preserve this record, develop future changes on development data, and freeze the final evaluation protocol before assessing the intended IDS defense contribution.
